# 01 — Dataset Audit

Medical Image Classification Challenge 2026.

Notebook này chỉ audit dữ liệu NORMAL/ và PNEUMONIA/, không train model và không sửa ảnh gốc.

Kiểm tra: số lượng ảnh, file hỏng, kích thước/aspect ratio, mode/channel, brightness/contrast, exact duplicates SHA-256, cross-label duplicates, ảnh mẫu và báo cáo CSV/JSON.

In [ ]:
from pathlib import Path
import hashlib, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = False
SEED = 2026
random.seed(SEED); np.random.seed(SEED)

PROJECT_ROOT = Path(r'D:\\medicine')
CLASS_DIRS = {'Normal': PROJECT_ROOT/'NORMAL', 'Pneumonia': PROJECT_ROOT/'PNEUMONIA'}
EXPECTED_COUNTS = {'Normal': 1299, 'Pneumonia': 3833}
OUTPUT_DIR = PROJECT_ROOT/'audit_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

print('Project root:', PROJECT_ROOT)
for label, path in CLASS_DIRS.items():
    print(f'{label:10s}: {path} | exists={path.exists()}')


## 1. Inventory và xác minh số lượng

In [ ]:
records = []
for label, folder in CLASS_DIRS.items():
    if not folder.exists(): raise FileNotFoundError(f'Không tìm thấy folder: {folder}')
    files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)
    for p in files:
        records.append({'label':label,'filename':p.name,'path':str(p),'extension':p.suffix.lower(),'file_size_bytes':p.stat().st_size})

df = pd.DataFrame(records)
counts = df['label'].value_counts().reindex(['Normal','Pneumonia'], fill_value=0)
print(counts); print('Total:', len(df))
for label, expected in EXPECTED_COUNTS.items():
    actual = int(counts[label])
    print(f"{'PASS' if actual == expected else 'CHECK'}: {label}: actual={actual}, expected={expected}")
imbalance_ratio = counts['Pneumonia']/max(counts['Normal'],1)
print(f'Pneumonia / Normal = {imbalance_ratio:.3f}:1')

fig, ax = plt.subplots(figsize=(6,4))
counts.plot(kind='bar', ax=ax); ax.set_title('Class distribution'); ax.set_ylabel('Images'); ax.set_xlabel('')
for i, value in enumerate(counts.values): ax.text(i,value,f'{value}\n({value/len(df):.1%})',ha='center',va='bottom')
plt.tight_layout(); plt.show()


## 2. Đọc ảnh, metadata, brightness/contrast và SHA-256

In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

def inspect_image(path):
    out = dict(read_ok=False,width=np.nan,height=np.nan,aspect_ratio=np.nan,mode=None,channels=np.nan,brightness=np.nan,contrast=np.nan,sha256=None,error=None)
    try:
        out['sha256'] = sha256_file(path)
        with Image.open(path) as img: img.verify()
        with Image.open(path) as img:
            w,h = img.size; mode = img.mode; channels = len(img.getbands())
            gray = img.convert('L'); gray.thumbnail((256,256)); arr = np.asarray(gray,dtype=np.float32)
            out.update(read_ok=True,width=int(w),height=int(h),aspect_ratio=float(w/h) if h else np.nan,mode=mode,channels=int(channels),brightness=float(arr.mean()),contrast=float(arr.std()))
    except Exception as e: out['error'] = repr(e)
    return out

rows=[]
for i,row in df.iterrows():
    rows.append({**row.to_dict(), **inspect_image(row['path'])})
    if (i+1)%250==0 or i+1==len(df): print(f'Scanned {i+1}/{len(df)}')

audit = pd.DataFrame(rows)
audit.to_csv(OUTPUT_DIR/'image_audit.csv',index=False)
bad = audit[~audit['read_ok']].copy(); bad.to_csv(OUTPUT_DIR/'corrupt_images.csv',index=False)
print('Corrupt/unreadable:',len(bad))
if len(bad): display(bad[['label','filename','error']].head(20))


## 3. Kích thước, aspect ratio, image mode và channel

In [ ]:
good = audit[audit['read_ok']].copy()
summary_by_class = good.groupby('label').agg(images=('filename','count'),width_min=('width','min'),width_median=('width','median'),width_max=('width','max'),height_min=('height','min'),height_median=('height','median'),height_max=('height','max'),aspect_median=('aspect_ratio','median'),brightness_mean=('brightness','mean'),contrast_mean=('contrast','mean')).round(3)
display(summary_by_class)
print('Image modes:'); display(pd.crosstab(good['label'],good['mode']))
print('Channels:'); display(pd.crosstab(good['label'],good['channels']))

for column,title in [('width','Image width'),('height','Image height'),('aspect_ratio','Aspect ratio W/H'),('brightness','Brightness'),('contrast','Contrast')]:
    fig,ax=plt.subplots(figsize=(7,4))
    for label in ['Normal','Pneumonia']:
        ax.hist(good.loc[good['label']==label,column].dropna(),bins=40,alpha=.5,label=label)
    ax.set_title(title); ax.legend(); plt.tight_layout(); plt.show()


## 4. Exact duplicate detection

Cùng SHA-256 = cùng bytes. Cross-label duplicate (Normal và Pneumonia cùng ảnh) là lỗi dữ liệu nghiêm trọng.

In [ ]:
hash_counts = good['sha256'].value_counts(); duplicate_hashes = hash_counts[hash_counts>1].index
exact_duplicates = good[good['sha256'].isin(duplicate_hashes)].sort_values(['sha256','label','filename']).copy()
exact_duplicates.to_csv(OUTPUT_DIR/'exact_duplicates.csv',index=False)
cross_label_hashes = [sha for sha,group in exact_duplicates.groupby('sha256') if group['label'].nunique()>1]
cross_label_duplicates = exact_duplicates[exact_duplicates['sha256'].isin(cross_label_hashes)].copy()
cross_label_duplicates.to_csv(OUTPUT_DIR/'cross_label_duplicates.csv',index=False)
print('Duplicate groups:',len(duplicate_hashes)); print('Images in duplicate groups:',len(exact_duplicates)); print('Cross-label duplicate groups:',len(cross_label_hashes))
if len(exact_duplicates): display(exact_duplicates[['sha256','label','filename']].head(30))
if len(cross_label_duplicates):
    print('WARNING: exact duplicate xuất hiện ở cả hai nhãn.'); display(cross_label_duplicates[['sha256','label','filename']])


## 5. Extension, filename và ảnh mẫu

In [ ]:
print('Extensions:'); display(pd.crosstab(audit['label'],audit['extension']))
duplicate_names = audit.groupby('filename').filter(lambda g:len(g)>1).sort_values('filename')
print('Same filename more than once:',len(duplicate_names))
if len(duplicate_names): display(duplicate_names[['label','filename','sha256']].head(30))

def show_random_samples(dataframe,label,n=8):
    subset=dataframe[(dataframe['label']==label)&dataframe['read_ok']]
    sample=subset.sample(n=min(n,len(subset)),random_state=SEED)
    cols=4; rows=int(np.ceil(len(sample)/cols)); fig,axes=plt.subplots(rows,cols,figsize=(12,3.2*rows)); axes=np.atleast_1d(axes).ravel()
    for ax in axes: ax.axis('off')
    for ax,(_,row) in zip(axes,sample.iterrows()):
        with Image.open(row['path']) as img: ax.imshow(img.convert('L'),cmap='gray')
        ax.set_title(f"{row['filename']}\n{int(row['width'])}x{int(row['height'])} | B={row['brightness']:.1f} C={row['contrast']:.1f}",fontsize=8); ax.axis('off')
    fig.suptitle(label); plt.tight_layout(); plt.show()

show_random_samples(good,'Normal'); show_random_samples(good,'Pneumonia')


## 6. Xuất audit summary và gates trước khi train

In [ ]:
summary = {
 'seed':SEED,'project_root':str(PROJECT_ROOT),'total_images':int(len(audit)),
 'class_counts':{k:int(v) for k,v in counts.to_dict().items()},
 'class_percentages':{k:float(v/len(audit)) for k,v in counts.to_dict().items()},
 'pneumonia_to_normal_ratio':float(imbalance_ratio),'readable_images':int(audit['read_ok'].sum()),'corrupt_images':int((~audit['read_ok']).sum()),
 'unique_exact_duplicate_groups':int(len(duplicate_hashes)),'images_in_exact_duplicate_groups':int(len(exact_duplicates)),'cross_label_duplicate_groups':int(len(cross_label_hashes)),
 'image_modes':{str(k):int(v) for k,v in good['mode'].value_counts(dropna=False).to_dict().items()},
 'width':{'min':int(good['width'].min()),'median':float(good['width'].median()),'max':int(good['width'].max())},
 'height':{'min':int(good['height'].min()),'median':float(good['height'].median()),'max':int(good['height'].max())},
 'brightness_mean_by_class':{k:float(v) for k,v in good.groupby('label')['brightness'].mean().to_dict().items()},
 'contrast_mean_by_class':{k:float(v) for k,v in good.groupby('label')['contrast'].mean().to_dict().items()}}
with open(OUTPUT_DIR/'audit_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2,ensure_ascii=False)
print(json.dumps(summary,indent=2,ensure_ascii=False)); print('Saved to:',OUTPUT_DIR)

checks={'expected_total_5132':len(audit)==5132,'expected_normal_1299':int(counts['Normal'])==1299,'expected_pneumonia_3833':int(counts['Pneumonia'])==3833,'all_images_readable':bool(audit['read_ok'].all()),'no_cross_label_exact_duplicates':len(cross_label_hashes)==0}
print('\nAUDIT GATES')
for name,passed in checks.items(): print(f"{'PASS' if passed else 'CHECK':5s} | {name}")
print('\nNext: review audit, then design leakage-resistant Train/Validation/Internal-Holdout split.')
